In [1]:
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import os
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 1. Model multimodalny
model_name = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_name)
processor = CLIPProcessor.from_pretrained(model_name)

# 2. Wczytaj obrazy
image_paths = ["./img/img1.jpg", "./img/img2.jpg", "./img/img3.jpg", "./img/img4.jpg", "./img/img5.jpg"]
images = [Image.open(p).convert("RGB") for p in image_paths]

# 3. Zapytania tekstowe
queries = [
    "dog on the grass",
    "animal in the house",
    "something to eat",
    "building in the city",
    "red car"
]

# 4. Embedowanie
with torch.no_grad():
    inputs = processor(text=queries, images=images, return_tensors="pt", padding=True)
    outputs = model(**inputs)

text_emb = outputs.text_embeds.numpy()
image_emb = outputs.image_embeds.numpy()

# 5. Cosine similarity — text vs każdy image
sim = cosine_similarity(text_emb, image_emb)

# 6. Wyniki
for i, q in enumerate(queries):
    best = np.argmax(sim[i])
    print(f"Query: {q}")
    print(f"Best image: {image_paths[best]} (score={sim[i][best]:.3f})")
    print("-" * 60)


c:\studies-master\deep-learning-in-text-processing\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Query: dog on the grass
Best image: ./img/img1.jpg (score=0.286)
------------------------------------------------------------
Query: animal in the house
Best image: ./img/img3.jpg (score=0.243)
------------------------------------------------------------
Query: something to eat
Best image: ./img/img4.jpg (score=0.227)
------------------------------------------------------------
Query: building in the city
Best image: ./img/img5.jpg (score=0.235)
------------------------------------------------------------
Query: red car
Best image: ./img/img2.jpg (score=0.242)
------------------------------------------------------------


In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset

model_name = "xlm-roberta-base"  # lub "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)  # 2 klasy: positive/negative


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
dataset = load_dataset("imdb")  # English dataset
train_texts = dataset['train']['text'][:500]  # dla demo można wziąć 500
train_labels = dataset['train']['label'][:500]

test_texts = dataset['test']['text'][:100]
test_labels = dataset['test']['label'][:100]

In [4]:
def preprocess(texts, labels):
    encodings = tokenizer(texts, truncation=True, padding=True)
    encodings['labels'] = labels
    return encodings


train_enc = preprocess(train_texts, train_labels)
test_enc = preprocess(test_texts, test_labels)

In [5]:
from torch.utils.data import Dataset

class SentimentDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
    def __len__(self):
        return len(self.encodings['input_ids'])
    def __getitem__(self, idx):
        return {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}

train_dataset = SentimentDataset(train_enc)
test_dataset = SentimentDataset(test_enc)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_steps=10,
    save_strategy="epoch",
    learning_rate=2e-5,
    seed=42,
    fp16=True if torch.cuda.is_available() else False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

c:\studies-master\deep-learning-in-text-processing\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


KeyboardInterrupt: 

In [ ]:
def predict(texts):
    enc = tokenizer(texts, truncation=True, padding=True, return_tensors="pt")
    with torch.no_grad():
        logits = model(**enc).logits
    return torch.argmax(logits, dim=-1).tolist()

english_samples = ["I loved this movie!", "It was terrible..."]
preds_eng = predict(english_samples)
print(preds_eng)  # np. [1,0] → 1=positive, 0=negative


[1, 1]


In [ ]:
polish_samples = ["Bardzo podobał mi się ten film!", "Był okropny..."]
preds_pl = predict(polish_samples)
print(preds_pl)


[1, 1]
